# Preparação das tabelas municipais do Atlas de Risco de Inundações — 2025

Este notebook reorganiza as tabelas extraídas das páginas 43 a 45 do Atlas em uma estrutura com duas colunas: `municipio` e `risco`. O objetivo é preparar os registros para possíveis análises descritivas e contextualização regional do projeto.


## Preparação e localização dos arquivos

A leitura e a reorganização dos CSVs de `dados/extraidos/` utilizam Pandas. O código pressupõe execução a partir de `notebooks/atlas/`. `Path.cwd().resolve().parents[1]` sobe dois níveis e define a raiz do projeto.


In [1]:
from pathlib import Path
import pandas as pd

In [2]:
RAIZ = Path.cwd().resolve().parents[1]

DIRETORIO = RAIZ / 'dados' / 'extraidos'

In [3]:
arquivos = sorted(DIRETORIO.glob('atlas_*.csv'))

print(f'Arquivos encontrados: {len(arquivos)}')

for arquivo in arquivos:
    print(arquivo.name)

Arquivos encontrados: 3
atlas_43_2.csv
atlas_44_2.csv
atlas_45_2.csv


## Página 43 — risco muito alto e alto

A tabela distribui os municípios em três colunas: a primeira corresponde ao risco muito alto; as duas seguintes, ao risco alto. Os rótulos de classificação aparecem na primeira linha dos dados e precisam ser separados dos nomes dos municípios.


In [4]:
df_43 = pd.read_csv(DIRETORIO / 'atlas_43_2.csv')

df_43


,Risco a Inundações dos municípios do Estado de Rio Grande do Sul\n(Classificação Final),Unnamed: 1,Unnamed: 2
0,Muito Alto (43),Alto (82),NaN
1,Alegrete,Agudo,Nova Palma
2,Alvorada,Arambaré,Nova Petrópolis
3,Barra do Guarita,Arroio do Meio,Osório
4,Barra do Ribeiro,Arroio do Sal,Paim Filho
5,Cachoeirinha,Arroio Grande,Pantano Grande
6,Campo Bom,Bagé,Pareci Novo
7,Canoas,Barra do Quaraí,Pedro Osório
8,Charqueadas,Barra Funda,Picada Café
9,Cruzeiro do Sul,Bento Gonçalves,Portão


In [5]:
coluna_0 = df_43[df_43.columns[0]]
coluna_1 = df_43[df_43.columns[1]]
coluna_2 = df_43[df_43.columns[2]]

coluna_2.dropna(inplace=True)

colunas = [coluna_0, coluna_1, coluna_2]

risco_muito_alto = coluna_0[0] 
risco_alto = coluna_1[0]

coluna_0 = coluna_0[1:]
coluna_1 = coluna_1[1:]

### Organização em registros municipais

Cada município é associado à sua classificação. As colunas são reunidas verticalmente, formando uma tabela com um município por linha. O resultado desta página contém 125 registros: 43 de risco muito alto e 82 de risco alto.


In [6]:
def processa_registro(registro, risco):
    return {
        'municipio': registro,
        'risco': risco
    }

df_43_processado = pd.DataFrame()

df_43_processado = pd.concat([
    pd.DataFrame(list(map(lambda x: processa_registro(x, risco_muito_alto), coluna_0))),
    pd.DataFrame(list(map(lambda x: processa_registro(x, risco_alto), coluna_1))),
    pd.DataFrame(list(map(lambda x: processa_registro(x, risco_alto), coluna_2)))
], ignore_index=True)

df_43_processado



,municipio,risco
0,Alegrete,Muito Alto (43)
1,Alvorada,Muito Alto (43)
2,Barra do Guarita,Muito Alto (43)
3,Barra do Ribeiro,Muito Alto (43)
4,Cachoeirinha,Muito Alto (43)
...,...,...
120,Turuçu,Alto (82)
121,Vale Verde,Alto (82)
122,Vale Real,Alto (82)
123,Venâncio Aires,Alto (82)


## Página 44 — risco médio

As três colunas contêm municípios da mesma classificação. A função `processa_df_44_45()` separa o rótulo de risco, desconsidera células vazias e reúne os nomes em uma única coluna. O resultado salvo apresenta 108 registros.


In [7]:
df_44 = pd.read_csv(DIRETORIO / 'atlas_44_2.csv')

df_44

,Risco a Inundações dos municípios do Estado de Rio Grande do Sul\n(Classificação Final),Unnamed: 1,Unnamed: 2
0,Médio (108),NaN,NaN
1,Ajuricaba,Eugênio de Castro,Quatro Irmãos
2,Alecrim,Fontoura Xavier,Quinze de Novembro
3,Alegria,Forquetinha,Relvado
4,Alto Alegre,Frederico Westphalen,Rio dos Índios
5,Amaral Ferrador,Gentil,Riozinho
6,Araricá,Gramado,Rodeio Bonito
7,Arroio do Padre,Guarani das Missões,Ronda Alta
8,Arroio do Tigre,Harmonia,Salvador do Sul
9,Barão de Cotegipe,Ivoti,Santa Maria do Herval


In [8]:
def processa_df_44_45(df):
    risco= df[df.columns[0]][0]

    coluna_0 = df[df.columns[0]][1:].reset_index(drop=True).dropna()
    coluna_1 = df[df.columns[1]][1:].reset_index(drop=True).dropna()
    coluna_2 = df[df.columns[2]][1:].reset_index(drop=True).dropna()

    df_processado = pd.DataFrame()

    df_processado = pd.concat([
        pd.DataFrame(list(map(lambda x: processa_registro(x, risco), coluna_0))),
        pd.DataFrame(list(map(lambda x: processa_registro(x, risco), coluna_1))),
        pd.DataFrame(list(map(lambda x: processa_registro(x, risco), coluna_2)))
    ], ignore_index=True)

    return df_processado


df_44_processado = processa_df_44_45(df_44)
df_44_processado

,municipio,risco
0,Ajuricaba,Médio (108)
1,Alecrim,Médio (108)
2,Alegria,Médio (108)
3,Alto Alegre,Médio (108)
4,Amaral Ferrador,Médio (108)
...,...,...
103,Vanini,Médio (108)
104,Vera Cruz,Médio (108)
105,Viadutos,Médio (108)
106,Viamão,Médio (108)


## Página 45 — risco baixo

Esta página utiliza a mesma organização da anterior. A mesma função de processamento produz 113 registros classificados como risco baixo.


In [9]:
df_45 = pd.read_csv(DIRETORIO / 'atlas_45_2.csv')

df_45

,Risco a Inundações dos municípios do Estado de Rio Grande do Sul\n(Classificação Final),Unnamed: 1,Unnamed: 2
0,Baixo (113),NaN,NaN
1,Alpestre,Erval Seco,Pouso Novo
2,Alto Feliz,Estrela Velha,Presidente Lucena
3,Ametista do Sul,Garibaldi,Progresso
4,Anta Gorda,Gaurama,Protásio Alves
5,Aratiba,Getúlio Vargas,Putinga
6,Arroio dos Ratos,Gramado dos Loureiros,Quevedos
7,Arvorezinha,Guaporé,Salvador das Missões
8,Áurea,Ibarama,Sananduva
9,Balneário Pinhal,Imigrante,Santa Bárbara do Sul


In [10]:
df_45_processado = processa_df_44_45(df_45)
df_45_processado

,municipio,risco
0,Alpestre,Baixo (113)
1,Alto Feliz,Baixo (113)
2,Ametista do Sul,Baixo (113)
3,Anta Gorda,Baixo (113)
4,Aratiba,Baixo (113)
...,...,...
108,União da Serra,Baixo (113)
109,Veranópolis,Baixo (113)
110,Vila Maria,Baixo (113)
111,Vista Gaúcha,Baixo (113)


## Consolidação e exportação

As três tabelas processadas são concatenadas, totalizando 346 registros, e exportadas para `dados/tratados/atlas_2025_processado.csv`. Os rótulos de risco são preservados, incluindo as quantidades entre parênteses presentes na fonte.


In [11]:
df_processado = pd.concat([df_43_processado, df_44_processado, df_45_processado], ignore_index=True)

df_processado.to_csv(RAIZ / 'dados' / 'tratados' / 'atlas_2025_processado.csv', index=False)

## Síntese e possíveis usos

| Classificação | Registros |
| --- | ---: |
| Muito alto | 43 |
| Alto | 82 |
| Médio | 108 |
| Baixo | 113 |
| **Total** | **346** |

A base pode apoiar comparações descritivas entre as classificações e contextualização municipal. Sua escala não permite atribuir diretamente o risco aos bairros de Porto Alegre.

A concordância das contagens com os rótulos não valida, isoladamente, os nomes extraídos ou a cobertura completa do Atlas. Uma possível etapa adicional é conferir os registros com o PDF e uma referência municipal. Este notebook ainda não contém verificações explícitas de duplicatas, padronização dos nomes ou cobertura.
